# Daily and weekly sales distributions

This notebook gives a compact overview of the processed demand series at their native aggregation levels. Each aggregation is analysed for both the regular dataset and the FCM dataset.

The unit of analysis is an `ARTIKEL_ID` x `MARKT_ID` series. A period is active when a row exists; a demand period has positive `ABVERKAUFTE_MENGE_KG`. The weekly data additionally contains the number of active, demand, and zero-sales days in each week.


In [55]:
from pathlib import Path
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
import matplotlib
matplotlib.use("Agg")
import duckdb
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)
GROUP_COLS = ["ARTIKEL_ID", "MARKT_ID"]
DEMAND_COL = "ABVERKAUFTE_MENGE_KG"
DATA_ROOT = next((p for p in [Path("../../data/processed"), Path("../data/processed"), Path("data/processed")] if p.exists()), None)
if DATA_ROOT is None: raise FileNotFoundError("Could not find data/processed from the notebook directory.")
DATASETS = {"daily": {"regular": DATA_ROOT / "transactions_dst_over_days", "fcm": DATA_ROOT / "transactions_dst_over_days_fcm"}, "weekly": {"regular": DATA_ROOT / "transactions_dst_over_weeks", "fcm": DATA_ROOT / "transactions_dst_over_weeks_fcm"}}
for aggregation in DATASETS:
    for variant, path in DATASETS[aggregation].items():
        if not list(path.glob("*.parquet")): raise FileNotFoundError(f"No parquet files found in {path}")
con = duckdb.connect()
con.execute("PRAGMA threads=4")


## Analysis helpers

The summaries below are generated with one function so daily and weekly results use the same definitions. The weekly `demand_day_share` is the share of active open days with positive sales, while `demand_period_share` is the share of active weeks with positive weekly demand.


In [56]:
def load_series_metrics(aggregation, variant, data_dir):
    path = str(data_dir / "*.parquet")
    if aggregation == "daily":
        query = f"""
        WITH rows AS (
            SELECT ARTIKEL_ID, MARKT_ID, CAST(COALESCE({DEMAND_COL}, 0) AS DOUBLE) AS demand, CAST(DATE AS DATE) AS period
            FROM read_parquet(?)
        )
        SELECT ARTIKEL_ID, MARKT_ID, COUNT(*)::INTEGER AS active_periods,
               SUM((demand > 0)::INTEGER)::INTEGER AS demand_periods,
               SUM((demand <= 0)::INTEGER)::INTEGER AS zero_periods,
               SUM(demand) AS total_demand, MIN(period) AS first_period, MAX(period) AS last_period,
               NULL::DOUBLE AS demand_day_share
        FROM rows GROUP BY ARTIKEL_ID, MARKT_ID
        """
    else:
        query = f"""
        SELECT ARTIKEL_ID, MARKT_ID, COUNT(*)::INTEGER AS active_periods,
               SUM((COALESCE({DEMAND_COL}, 0) > 0)::INTEGER)::INTEGER AS demand_periods,
               SUM((COALESCE({DEMAND_COL}, 0) <= 0)::INTEGER)::INTEGER AS zero_periods,
               SUM(COALESCE({DEMAND_COL}, 0)) AS total_demand,
               MIN(CAST(DATE AS DATE)) AS first_period, MAX(CAST(DATE AS DATE)) AS last_period,
               SUM(COALESCE(demand_days_in_week, 0)) / NULLIF(SUM(COALESCE(active_days_in_week, 0)), 0) AS demand_day_share
        FROM read_parquet(?) GROUP BY ARTIKEL_ID, MARKT_ID
        """
    result = con.execute(query, [path]).fetchdf()
    result["aggregation"], result["variant"] = aggregation, variant
    result["demand_period_share"] = result["demand_periods"] / result["active_periods"]
    if aggregation == "daily": result["demand_day_share"] = result["demand_period_share"]
    result["zero_period_share"] = result["zero_periods"] / result["active_periods"]
    result["dataset"] = aggregation.title() + " / " + variant.title()
    return result

metrics = pd.concat([load_series_metrics(a, v, p) for a, variants in DATASETS.items() for v, p in variants.items()], ignore_index=True)
DATASET_ORDER = ["Daily / Regular", "Daily / Fcm", "Weekly / Regular", "Weekly / Fcm"]
metrics["dataset"] = pd.Categorical(metrics["dataset"], categories=DATASET_ORDER, ordered=True)
metrics.head()


,ARTIKEL_ID,MARKT_ID,active_periods,demand_periods,zero_periods,total_demand,first_period,last_period,demand_day_share,aggregation,variant,demand_period_share,zero_period_share,dataset
0,316646,1350009,1521,1386,135,2510.613,2021-07-01,2026-06-30,0.911243,daily,regular,0.911243,0.088757,Daily / Regular
1,230100,1350009,1520,967,553,515.713,2021-07-02,2026-06-30,0.636184,daily,regular,0.636184,0.363816,Daily / Regular
2,317207,1350009,1519,361,1158,774.167,2021-07-03,2026-06-30,0.237656,daily,regular,0.237656,0.762344,Daily / Regular
3,1206404,1350009,796,3,793,1.202,2023-11-13,2026-06-30,0.003769,daily,regular,0.003769,0.996231,Daily / Regular
4,1061313,1350009,1330,149,1181,39.994,2022-02-11,2026-06-30,0.112030,daily,regular,0.112030,0.887970,Daily / Regular


# Daily analysis

Daily rows represent the active open-day periods created by the data preparation pipeline.


In [57]:
daily = metrics[metrics["aggregation"] == "daily"].copy()
daily_overview = (daily.groupby("dataset", observed=True).agg(series=("ARTIKEL_ID", "size"), stores=("MARKT_ID", "nunique"), products=("ARTIKEL_ID", "nunique"), active_days=("active_periods", "sum"), demand_days=("demand_periods", "sum"), zero_days=("zero_periods", "sum"), total_demand_kg=("total_demand", "sum")).reset_index())
daily_overview["demand_day_share"] = daily_overview["demand_days"] / daily_overview["active_days"]
display(daily_overview.style.format({"total_demand_kg":"{:,.1f}", "demand_day_share":"{:.1%}"}))


,dataset,series,stores,products,active_days,demand_days,zero_days,total_demand_kg,demand_day_share
0,Daily / Regular,199,1,199,242508,47411,195097,"66,408.0",19.6%
1,Daily / Fcm,6,1,6,1434,228,1206,79.6,15.9%


In [58]:
def distribution_table(frame, columns):
    rows = []
    for dataset, group in frame.groupby("dataset", observed=True):
        for column in columns:
            q = group[column].dropna().quantile([.25, .5, .75, .95])
            rows.append({"dataset":dataset, "measure":column, "p25":q.loc[.25], "median":q.loc[.5], "p75":q.loc[.75], "p95":q.loc[.95]})
    return pd.DataFrame(rows)

quantile_format = {"p25":"{:.2f}", "median":"{:.2f}", "p75":"{:.2f}", "p95":"{:.2f}"}
display(distribution_table(daily, ["active_periods", "demand_periods", "zero_periods", "demand_day_share"]).style.format(quantile_format))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=daily, x="active_periods", hue="dataset", bins=30, element="step", stat="density", common_norm=False, ax=axes[0])
sns.histplot(data=daily, x="demand_periods", hue="dataset", bins=30, element="step", stat="density", common_norm=False, ax=axes[1])
axes[0].set_title("Active-day counts per series"); axes[1].set_title("Demand-day counts per series"); plt.tight_layout()


,dataset,measure,p25,median,p75,p95
0,Daily / Regular,active_periods,1084.50,1429.00,1519.00,1521.00
1,Daily / Regular,demand_periods,8.00,59.00,369.50,1096.00
2,Daily / Regular,zero_periods,722.00,1088.00,1356.00,1481.20
3,Daily / Regular,demand_day_share,0.01,0.06,0.28,0.75
4,Daily / Fcm,active_periods,159.25,272.50,325.75,354.25
5,Daily / Fcm,demand_periods,3.25,16.00,73.75,99.50
6,Daily / Fcm,zero_periods,110.25,255.50,259.75,317.25
7,Daily / Fcm,demand_day_share,0.02,0.07,0.24,0.61


## Daily binned count distributions

Bins make the long-tailed period counts easier to compare than raw histograms.


In [59]:
def binned_counts(frame, columns, bins, labels):
    parts = []
    for column in columns:
        part = frame[["dataset", column]].copy(); part["measure"] = column
        part["bin"] = pd.cut(part[column], bins=bins, labels=labels, right=True)
        counts = part.groupby(["dataset", "measure", "bin"], observed=False).size().rename("series").reset_index()
        counts["share"] = counts["series"] / counts.groupby(["dataset", "measure"], observed=False)["series"].transform("sum")
        parts.append(counts)
    return pd.concat(parts, ignore_index=True)
DAY_BINS = [-1, 0, 1, 7, 14, 30, 60, 90, 180, 365, 730, np.inf]
DAY_LABELS = ["0", "1", "2-7", "8-14", "15-30", "31-60", "61-90", "91-180", "181-365", "366-730", ">730"]
daily_bins = binned_counts(daily, ["active_periods", "demand_periods"], DAY_BINS, DAY_LABELS)
display(daily_bins.pivot_table(index=["measure", "bin"], columns="dataset", values="share", observed=False).style.format("{:.1%}"))


The `top_store_products` table summarizes where demand is concentrated. It first sums demand for each store-product pair (`MARKT_ID`, `ARTIKEL_ID`), then ranks stores by the number of products with positive demand and by total demand. `top_product_ids` lists the three product IDs with the highest total demand in that store. The `top_store_products` table summarizes where demand is concentrated.


In [60]:
def top_store_products(frame, n=10):
    store_product = frame.groupby(["MARKT_ID", "ARTIKEL_ID"], observed=True).agg(demand=("total_demand", "sum")).reset_index()
    top_stores = (store_product.groupby("MARKT_ID", as_index=False).agg(products_with_demand=("ARTIKEL_ID", "nunique"), total_demand=("demand", "sum")).sort_values(["products_with_demand", "total_demand"], ascending=False).head(n))
    top_products = store_product.sort_values(["MARKT_ID", "demand"], ascending=[True, False]).groupby("MARKT_ID", as_index=False).head(3)
    ids = (top_products.assign(product_id=top_products["ARTIKEL_ID"].astype(str)).groupby("MARKT_ID")["product_id"].agg(lambda values: ", ".join(values)).rename("top_product_ids"))
    return top_stores.merge(ids, on="MARKT_ID", how="left")
for dataset, group in daily.groupby("dataset", observed=True):
    print(dataset); display(top_store_products(group))


Daily / Regular


,MARKT_ID,products_with_demand,total_demand,top_product_ids
0,1350009,199,66408.021248,"316623, 329525, 317045"


Daily / Fcm


,MARKT_ID,products_with_demand,total_demand,top_product_ids
0,1350009,6,79.5752,"1382768, 1413121, 1382858"


# Weekly analysis

Weekly rows represent active weeks. The weekly data retains daily diagnostics, so this section reports both the share of weeks with demand and the share of active days with demand inside those weeks.


In [61]:
weekly = metrics[metrics["aggregation"] == "weekly"].copy()
weekly_overview = (weekly.groupby("dataset", observed=True).agg(series=("ARTIKEL_ID", "size"), stores=("MARKT_ID", "nunique"), products=("ARTIKEL_ID", "nunique"), active_weeks=("active_periods", "sum"), demand_weeks=("demand_periods", "sum"), zero_weeks=("zero_periods", "sum"), total_demand_kg=("total_demand", "sum"), demand_day_share=("demand_day_share", "mean")).reset_index())
weekly_overview["demand_week_share"] = weekly_overview["demand_weeks"] / weekly_overview["active_weeks"]
display(weekly_overview.style.format({"total_demand_kg":"{:,.1f}", "demand_day_share":"{:.1%}", "demand_week_share":"{:.1%}"}))


,dataset,series,stores,products,active_weeks,demand_weeks,zero_weeks,total_demand_kg,demand_day_share,demand_week_share
0,Weekly / Regular,199,1,199,41883,15688,26195,"66,408.0",17.5%,37.5%
1,Weekly / Fcm,6,1,6,254,88,166,79.6,19.3%,34.6%


In [62]:
display(distribution_table(weekly, ["active_periods", "demand_periods", "zero_periods", "demand_day_share", "demand_period_share"]).style.format(quantile_format))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=weekly, x="active_periods", hue="dataset", bins=30, element="step", stat="density", common_norm=False, ax=axes[0])
sns.histplot(data=weekly, x="demand_periods", hue="dataset", bins=30, element="step", stat="density", common_norm=False, ax=axes[1])
axes[0].set_title("Active-week counts per series"); axes[1].set_title("Demand-week counts per series"); plt.tight_layout()


,dataset,measure,p25,median,p75,p95
0,Weekly / Regular,active_periods,188.00,247.00,262.00,262.00
1,Weekly / Regular,demand_periods,5.50,34.00,161.50,257.10
2,Weekly / Regular,zero_periods,45.00,142.00,216.50,245.10
3,Weekly / Regular,demand_day_share,0.01,0.06,0.28,0.75
4,Weekly / Regular,demand_period_share,0.04,0.16,0.66,0.98
5,Weekly / Fcm,active_periods,28.75,48.00,57.50,62.25
6,Weekly / Fcm,demand_periods,1.75,11.00,20.25,37.50
7,Weekly / Fcm,zero_periods,13.25,26.00,41.75,53.25
8,Weekly / Fcm,demand_day_share,0.02,0.07,0.24,0.61
9,Weekly / Fcm,demand_period_share,0.07,0.22,0.60,0.86


## Weekly binned count distributions

In [63]:
WEEK_BINS = [-1, 0, 1, 4, 8, 13, 26, 52, 104, 156, np.inf]
WEEK_LABELS = ["0", "1", "2-4", "5-8", "9-13", "14-26", "27-52", "53-104", "105-156", ">156"]
weekly_bins = binned_counts(weekly, ["active_periods", "demand_periods"], WEEK_BINS, WEEK_LABELS)
display(weekly_bins.pivot_table(index=["measure", "bin"], columns="dataset", values="share", observed=False).style.format("{:.1%}"))
for dataset, group in weekly.groupby("dataset", observed=True):
    print(dataset); display(top_store_products(group))


Weekly / Regular


,MARKT_ID,products_with_demand,total_demand,top_product_ids
0,1350009,199,66408.021248,"316623, 329525, 317045"


Weekly / Fcm


,MARKT_ID,products_with_demand,total_demand,top_product_ids
0,1350009,6,79.5752,"1382768, 1413121, 1382858"


## Reading the results

Compare zero-period shares with demand-period shares to quantify intermittent demand within active observations. For weekly data, compare `demand_day_share` with `demand_week_share` to see how weekly aggregation hides daily zero-sales periods. The store tables rank stores by the number of products with positive demand and include the top product IDs by total demand.
